# 11 — Evaluación interna agrupada

Comparación pareada del baseline clínico frente a prevalencia para Sepsis-3 a 6 horas. Se usan pacientes completos como unidad de bootstrap. Solo se abren `development` y `validation`; el test permanece cerrado. Las cifras del demo son controles técnicos.

In [ ]:
from pathlib import Path
import json, shutil, subprocess, sys, tempfile
import pandas as pd
from IPython.display import SVG, display
PROJECT_ROOT=Path.cwd().parent if Path.cwd().name=='notebooks' else Path.cwd()
sys.path.insert(0,str(PROJECT_ROOT/'src')) if str(PROJECT_ROOT/'src') not in sys.path else None
sys.path.insert(0,str(PROJECT_ROOT)) if str(PROJECT_ROOT) not in sys.path else None
from mimic_sepsis.artifacts import ArtifactStore, ArtifactValidationError
from mimic_sepsis.evaluation import patient_cluster_bootstrap_comparison, percentile_intervals
from mimic_sepsis.modeling import assemble_modeling_table, binary_metrics, equal_patient_weights, grouped_cross_validation, grouped_prevalence_cross_validation, make_logistic_pipeline, patient_weighted_event_rate
from scripts.build_demo_sofa_incremental import canonical_config
artifact_config=canonical_config(); model_config=json.loads((PROJECT_ROOT/'config/modeling.json').read_text()); evaluation_config=json.loads((PROJECT_ROOT/'config/evaluation.json').read_text())
valid=[]
for path in sorted((PROJECT_ROOT/'data/derived/sofa').glob('*/60_features/sepsis3_development_features.manifest.json')):
    try:
        manifest=ArtifactStore(path.parent).validate('sepsis3_development_features',expected_config=artifact_config); valid.append((manifest.created_at_utc,path.parents[1]))
    except (FileNotFoundError,ArtifactValidationError): pass
if not valid: raise RuntimeError('Ejecute primero los notebooks 00–10.')
RUN_ROOT=sorted(valid,key=lambda x:(x[0],str(x[1])))[-1][1]
landmark_store=ArtifactStore(RUN_ROOT/'50_landmarks'); feature_store=ArtifactStore(RUN_ROOT/'60_features')
print(f'Ejecución validada: {RUN_ROOT.name}')

## Predicciones comparables

In [ ]:
def load(partition):
    return assemble_modeling_table(landmark_store.read_dataframe(f'sepsis3_{partition}_landmarks',expected_config=artifact_config),feature_store.read_dataframe(f'sepsis3_{partition}_features',expected_config=artifact_config),horizon_hours=model_config['primary_horizon_hours'])
development=load('development'); validation=load('validation')
columns=model_config['clinical_baseline_features']; folds=model_config['cross_validation_folds']; seed=model_config['seed']
reference_oof,_=grouped_prevalence_cross_validation(development,folds=folds,seed=seed)
pipeline=make_logistic_pipeline(columns,c=model_config['logistic_c'],seed=seed)
candidate_oof,_=grouped_cross_validation(development,pipeline,columns,folds=folds,seed=seed)
keys=['subject_id','hadm_id','stay_id','landmark_time','fold','outcome']
oof=reference_oof.rename(columns={'probability':'reference'}).merge(candidate_oof.rename(columns={'probability':'candidate'}),on=keys,validate='one_to_one')
pipeline.fit(development[columns],development.outcome,model__sample_weight=equal_patient_weights(development))
validation_predictions=validation[['subject_id','hadm_id','stay_id','landmark_time','outcome']].copy()
validation_predictions['reference']=patient_weighted_event_rate(development)
validation_predictions['candidate']=pipeline.predict_proba(validation[columns])[:,1]
point_metrics=[]
for sample,frame in [('development_oof',oof),('validation',validation_predictions)]:
    for model in ('reference','candidate'):
        point_metrics.append({'sample':sample,'model':model,**binary_metrics(frame.outcome,frame[model])})
display(pd.DataFrame(point_metrics))

## Bootstrap pareado por paciente

In [ ]:
bootstraps=[]; intervals=[]
for offset,(sample,frame) in enumerate([('development_oof',oof),('validation',validation_predictions)]):
    boot=patient_cluster_bootstrap_comparison(frame,frame.reference,frame.candidate,replicates=evaluation_config['bootstrap_replicates'],seed=evaluation_config['seed']+offset)
    boot['sample']=sample; bootstraps.append(boot)
    summary=percentile_intervals(boot.drop(columns='sample'),confidence_level=evaluation_config['confidence_level']); summary['sample']=sample; intervals.append(summary)
intervals=pd.concat(intervals,ignore_index=True)
intervals['direction']=intervals.metric.map(lambda x:'Mayor favorece candidato' if x in ('delta_auroc','delta_auprc') else 'Menor favorece candidato')
display(intervals)
if (intervals.successful_replicates < 0.9*evaluation_config['bootstrap_replicates']).any(): print('ADVERTENCIA: muchos remuestreos sin ambas clases; incertidumbre no fiable en el demo.')

## Diferencias e incertidumbre con ggplot2

In [ ]:
rscript=shutil.which('Rscript')
if not rscript: raise RuntimeError('Rscript no está disponible.')
with tempfile.TemporaryDirectory() as tmp:
    tmp=Path(tmp); csv=tmp/'intervals.csv'; svg=tmp/'intervals.svg'; script=tmp/'plot.R'; intervals.to_csv(csv,index=False)
    script.write_text("""args <- commandArgs(trailingOnly=TRUE)
suppressPackageStartupMessages(library(ggplot2))
d <- read.csv(args[1]); d$metric <- factor(d$metric,levels=rev(unique(d$metric)))
p <- ggplot(d,aes(estimate,metric,colour=sample)) + geom_vline(xintercept=0,linetype=2,colour='grey50') + geom_errorbarh(aes(xmin=lower,xmax=upper),height=.18,position=position_dodge(width=.45)) + geom_point(position=position_dodge(width=.45),size=2) + facet_wrap(~direction,scales='free_x') + labs(title='Diferencia: baseline clínico menos prevalencia',subtitle='IC percentil por bootstrap de pacientes completos; demo no inferencial',x='Diferencia de métrica',y=NULL,colour='Muestra') + theme_minimal(base_size=11)
ggsave(args[2],p,width=10,height=5.5,device=grDevices::svg)
""")
    result=subprocess.run([rscript,str(script),str(csv),str(svg)],capture_output=True,text=True)
    if result.returncode: raise RuntimeError(result.stderr)
    display(SVG(filename=str(svg)))

## Interpretación

El remuestreo conserva juntos todos los landmarks de cada paciente y compara ambos modelos sobre las mismas réplicas. Con los pocos eventos del demo, intervalos amplios o réplicas sin ambas clases son esperables. No se elige modelo ni se abre test; la evaluación definitiva exige MIMIC-IV completo y el protocolo congelado.